In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13.0" peft accelerate bitsandbytes datasets

In [ ]:
DATASET_PATH = "/content/spl_final.jsonl"
lines = open(DATASET_PATH).readlines()
print(f"Dataset lines: {len(lines)}")

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=768,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import load_dataset

EOS = tokenizer.eos_token

def format_sample(row):
    return {
        "text": (
            "<|im_start|>system\n"
            + row["instruction"]
            + "<|im_end|>\n"
            + "<|im_start|>user\n"
            + row["input"]
            + "<|im_end|>\n"
            + "<|im_start|>assistant\n"
            + row["output"]
            + "<|im_end|>"
        )
    }

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(format_sample, remove_columns=dataset.column_names)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)}  Eval: {len(eval_dataset)}")

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling

tokenizer.padding_side = "right"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=768,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=SFTConfig(
        max_seq_length=768,
        dataset_num_proc=1,
        packing=False,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        warmup_ratio=0.1,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        output_dir="/content/qwen-spl",
        report_to="none",
        seed=42,
    ),
)

print(f"GPU: {torch.cuda.get_device_properties(0).name}")
trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

prompt = """<|im_start|>system
You are a reasoning assistant. Context arrives in Symbolic Prompt Language.
Read the structure directly. Respond precisely and do not invent missing facts.<|im_end|>
<|im_start|>user
[CTX]
task=deploy_flask_app deadline=T+1
pref=[nginx,systemd]
!pref=[docker,managed_cloud]
hw=2gb_ram scope=self_hosted
q_hist=[flask,gunicorn,ubuntu]
[/CTX]
[Q] How do I keep the app running after reboot?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
_ = model.generate(
    **inputs,
    streamer=streamer,
    max_new_tokens=200,
    temperature=0.1,
    do_sample=True,
    eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/qwen-spl-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen-spl-lora")

model.save_pretrained_gguf(
    "/content/drive/MyDrive/qwen-spl-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print("Saved to Google Drive.")